# E2E 02 -- Full Pipeline (Builder + Query)

End-to-end verification of the complete pipeline:
1. Builder Graph: documents + DDL -> Knowledge Graph
2. Query Graph: natural-language questions -> grounded answers


In [ ]:
from __future__ import annotations
import sys, os
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

os.environ.setdefault('LOG_LEVEL', 'WARNING')
os.environ.setdefault('NEO4J_URI',  'bolt://localhost:7687')
os.environ.setdefault('NEO4J_USER', 'neo4j')
os.environ.setdefault('NEO4J_PASSWORD', 'your_password')
os.environ.setdefault('LMSTUDIO_BASE_URL', 'http://localhost:1234/v1')
os.environ.setdefault('LLM_MODEL_REASONING',  'local-model')
os.environ.setdefault('LLM_MODEL_EXTRACTION', 'local-model')
os.environ.setdefault('LLM_MAX_TOKENS_EXTRACTION', '16384')

fixtures_dir = repo_root / 'tests' / 'fixtures'
doc_paths = [
    fixtures_dir / 'smoke' / 'business_glossary_smoke.txt',
    fixtures_dir / 'smoke' / 'data_dictionary_smoke.txt',
]
ddl_paths = [fixtures_dir / 'smoke' / 'smoke_schema.sql']

print(f'Repository root : {repo_root}')


In [ ]:
import time
from src.graph.builder_graph import run_builder

print('Phase 1: Builder Graph')
start = time.time()

builder_state = run_builder(
    raw_documents=doc_paths,
    ddl_paths=ddl_paths,
    production=False,
)

elapsed = time.time() - start
completed = builder_state.get('completed_tables', [])
print(f'  Completed in {elapsed:.1f} s | Tables completed: {len(completed)}')


In [ ]:
from src.generation.query_graph import run_query

questions = [
    'What are the main business entities in the domain?',
    'Which table stores customer information?',
    'How are customers and orders related?',
]

print('Phase 2: Query Graph')
print('-' * 50)

results = []
for question in questions:
    result = run_query(question)
    results.append({'question': question, 'final_answer': result.get('final_answer', '')})


In [ ]:
for item in results:
    assert item['final_answer'], f"Empty answer for: {item['question']}"

print('Query Graph Results')
print('=' * 60)
for i, item in enumerate(results, 1):
    answer = item['final_answer']
    if len(answer) > 300:
        answer = answer[:300] + ' [truncated]'
    print(f"[{i}] {item['question']}")
    print(f'    {answer}')
    print()

print('All assertions passed.')
